# 7장 GraphRAG: 관계를 묻다
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제2부 RAG — 검색으로 지식을 더하다
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
!pip install -q google-genai networkx

### API 키와 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.0-flash"   # 모델 갱신 시 이 한 줄만 변경

## 예제 7-1. 구조화 출력으로 트리플 추출

In [ ]:
import json

schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "주어": {"type": "string"},
            "관계": {"type": "string"},
            "대상": {"type": "string"},
        },
        "required": ["주어", "관계", "대상"],
    },
}
text = "이수미는 배재대학교에서 가르친다. 배재대학교는 대전에 있다."
res = client.models.generate_content(
    model=GEMINI_FLASH,
    contents=f"다음 문장에서 (주어, 관계, 대상) 트리플을 모두 뽑아라.\n{text}",
    config={
        "response_mime_type": "application/json",
        "response_schema": schema,
        "temperature": 0,
    },
)
for t in json.loads(res.text):
    print(t["주어"], "-", t["관계"], "->", t["대상"])

## 예제 7-2. 그래프 다중 홉 추적

In [ ]:
triples = [
    ("이수미", "소속", "배재대학교"),
    ("배재대학교", "위치", "대전"),
    ("대전", "포함", "대한민국"),
]

graph = {}
for s, r, o in triples:
    graph.setdefault(s, []).append((r, o))

def hop(start, depth=2):
    node, path = start, [start]
    for _ in range(depth):
        if node not in graph:
            break
        relation, target = graph[node][0]
        path.append(f"-{relation}-> {target}")
        node = target
    return " ".join(path)

print(hop("이수미"))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 7장 노트북